# What moves my HRV — the reasoning

This notebook walks through the same pipeline that generates `public/data/*.json` for the app
(`pipeline/load.py` → `quality.py` → `correlations.py` → `alcohol.py` → `bins.py`). Nothing here
is re-derived differently for the notebook — it imports the same functions, so if this notebook
and the app ever disagree, the pipeline has a bug.

Three findings drive the whole project:

1. **The cost of a drinking night is one night, not a weekend** — recovery dips hard the morning
   after and is back near baseline by the next one.
2. **Sleep has a plateau at 8–8.5h, not a straight-line benefit** — more sleep past that band
   doesn't buy more HRV.
3. **The model explains about a quarter of day-to-day HRV variance (R² ≈ 0.26)** — the rest is
   unmeasured, and the app says so rather than implying more precision than the data supports.


In [1]:
from pipeline.load import build_daily_table
from pipeline.quality import build_quality_report

daily, report = build_daily_table()
quality = build_quality_report()
print(f"Loaded {len(daily)} days.")
print()
print(f"Date range: {quality['date_range']['start']} -> {quality['date_range']['end']}")
print(f"Physio dedupe: {quality['dedupe']['physio_rows_raw']} raw rows -> "
      f"{quality['dedupe']['physio_rows_after_dedupe']} rows "
      f"({quality['dedupe']['physio_dup_dates_collapsed']} nap-duplicate dates collapsed)")
print(f"Rule: {quality['dedupe']['rule']}")


Date range: 2023-09-05 -> 2026-09-16
Days in joined table: 831
Physio dedupe: 1027 raw rows -> 196 duplicate dates collapsed -> 831 rows
Rule: Per date, keep the row with the highest Recovery score % (non-null beats null). Naps create the duplicate cycles.

Missing recovery/HRV/RHR: 32 days (3.9%)
Missing sleep consistency: 56 days

Journal question coverage:
                                question  n_answered  n_yes  sufficient
              Have any alcoholic drinks?         509    121        True
                        Shared your bed?         507     17       False
                    Feeling sick or ill?         507     13       False
                      Consumed caffeine?         415      5       False
          Viewed a screen device in bed?         509      5       False
                Have an injury or wound?         415      0       False
Read (non-screened device) while in bed?         507      0       False
     Took prescription sleep medication?         415      0   

## 1. Data quality, dealt with explicitly

Naps create a second same-day cycle in `physiological_cycles.csv` — 196 of 1,027 raw rows are
duplicate dates. The dedupe rule (keep the highest `Recovery score %` per date) lives once in
`pipeline/load.py` and every downstream module reuses the same joined table, so this rule can't
drift between analyses.

Missing values are never imputed. A day with no recovery/HRV/RHR reading (watch not worn or not
charged) is dropped from whatever analysis needs that field, and the drop is counted — see
`quality.json` for the exact numbers, which also drive the footer in the app.

Journal coverage is uneven and mostly too thin to use — see the table below. Only the alcohol
question has enough "yes" answers to support a claim.

In [1]:
import pandas as pd
pd.DataFrame(quality['journal']['questions'])

Date range: 2023-09-05 -> 2026-09-16
Days in joined table: 831
Physio dedupe: 1027 raw rows -> 196 duplicate dates collapsed -> 831 rows
Rule: Per date, keep the row with the highest Recovery score % (non-null beats null). Naps create the duplicate cycles.

Missing recovery/HRV/RHR: 32 days (3.9%)
Missing sleep consistency: 56 days

Journal question coverage:
                                question  n_answered  n_yes  sufficient
              Have any alcoholic drinks?         509    121        True
                        Shared your bed?         507     17       False
                    Feeling sick or ill?         507     13       False
                      Consumed caffeine?         415      5       False
          Viewed a screen device in bed?         509      5       False
                Have an injury or wound?         415      0       False
Read (non-screened device) while in bed?         507      0       False
     Took prescription sleep medication?         415      0   

## 2. Univariate correlations with HRV

A scan, not a claim — several of the strongest correlations here are deliberately excluded from
the app because of what the reasoning behind them reveals, not because the number is small.
Resting heart rate (r ≈ −0.89) is the standout case: it's the strongest number in the dataset
and the least useful, because RHR and HRV are two readings of the same autonomic state in the
same sleep window.

In [1]:
from pipeline.correlations import build_model_report
model = build_model_report()

uni = pd.DataFrame(model['univariate_correlations'])[['label', 'r', 'p', 'n', 'verdict']]
uni

                label      r             p   n          verdict
   Resting heart rate -0.888 2.377564e-271 799          exclude
  Sleep performance %  0.372  1.069454e-27 799     keep_context
     Respiratory rate -0.337  1.287652e-22 799 signal_not_lever
    Light sleep hours  0.303  1.999637e-18 799     keep_context
    Total sleep hours  0.237  1.250154e-11 799    primary_lever
   Sleep efficiency %  0.211  1.582400e-09 799     keep_context
     Deep (SWS) hours -0.191  5.758886e-08 799 counterintuitive
       Blood oxygen %  0.180  3.268380e-07 798     keep_context
Day strain (same day)  0.134  1.495830e-04 798  wrong_direction
           Sleep debt -0.123  4.781969e-04 799     weak_context
  Sleep consistency %  0.120  8.311507e-04 775       weak_lever
     Prior-day strain -0.073  4.037807e-02 798  secondary_lever
            REM hours -0.052  1.445278e-01 799  no_relationship

## 3. The multivariate model, and why respiratory rate stays in it

Restricting to actionable levers (sleep hours, alcohol, sleep consistency, prior-day strain,
skin temperature) plus respiratory rate as a covariate gives R² ≈ 0.26. Sleep and alcohol are
the two variables that survive as significant, correct-direction levers — consistency and
prior-day strain drop out once sleep and alcohol are controlled, meaning their univariate
correlations were mostly sleep in disguise.

In [1]:
primary = model['primary_model']
print(f"n={primary['n']}  R^2={primary['r_squared']}")
print(f"respiratory rate held at {primary['resp_rate_held_at']} rpm (not user-controlled)")
print()
pd.DataFrame(primary['coefficients']).T

n=775  R^2=0.258
intercept=426.49
respiratory rate held at 15.9 rpm, skin temp held at 34.9 C

                      coef_ms         p significant
sleep_hours              5.97       0.0        True
alcohol                -27.72       0.0        True
resp_rate              -14.98       0.0        True
sleep_consistency_pct    0.14  0.047234        True
prior_day_strain        -0.14  0.556246       False
skin_temp_c             -3.54  0.116187       False

### Why respiratory rate can't just be dropped

Fitting the same rows with and without a respiratory-rate term shows it *mediates* part of the
alcohol effect: drop it, and the alcohol coefficient inflates, because alcohol raises
respiratory rate and — without an rpm term to absorb that — the pathway gets re-attributed to
the alcohol dummy instead. That's why respiratory rate stays in the fitted model as a covariate
even though it's never a slider in the app.

In [1]:
rows = []
for name in ['with_respiratory_rate', 'without_respiratory_rate', 'sleep_and_alcohol_only']:
    v = model['model_variants'][name]
    rows.append({
        'model': name, 'n': v['n'], 'r_squared': v['r_squared'],
        'alcohol_coef_ms': v['coefficients'].get('alcohol', {}).get('coef_ms'),
        'sleep_coef_ms': v['coefficients'].get('sleep_hours', {}).get('coef_ms'),
    })
pd.DataFrame(rows)

                   model   n  r_squared  alcohol_coef_ms  sleep_coef_ms
   with_respiratory_rate 775      0.258           -27.72           5.97
without_respiratory_rate 775      0.193           -32.25           6.23
  sleep_and_alcohol_only 775      0.185           -32.53           6.58

## 4. The alcohol effect, standalone

In [1]:
model['alcohol_contrast']

{
  "hrv_alcohol": 88.3,
  "n_alcohol": 119,
  "hrv_sober": 127.6,
  "n_sober": 386,
  "cohens_d": -1.47,
  "p": 1.303305069131725e-28
}

## 5. The alcohol recovery curve

Day 0 is the *same* cycle as the drinking night, not the day after — WHOOP attributes a cycle's
recovery score to the sleep ending that morning. The dip is sharp and entirely gone by day +1:
no multi-day cascade.

In [1]:
from pipeline.alcohol import build_alcohol_report
alcohol = build_alcohol_report()

pd.DataFrame(alcohol['trajectory'])

 day   n  mean_recovery  mean_hrv  pct_ge_80
  -2  91           65.0     113.9       34.1
  -1  77           65.7     114.5       31.2
   0 119           46.8      88.3        8.4
   1  88           68.1     116.9       39.8
   2  97           67.0     115.7       37.1
   3  97           66.2     116.6       38.1
   4  92           69.1     119.1       41.3

Base rate, 80+ recovery on sober days: 43.8% (n=388)
Days to 80+ after a drink: median 2, 23.1% never within 5 days

Consecutive drinking nights: {'singles': 78, 'doubles': 14, 'triples': 5, 'four_plus': 0}

80+ recovery is a minority outcome even on sober nights, which is why the app never builds a
"days to 80" countdown — reaching 80+ takes a median of ~2 days after a drink mostly because
that's how many rolls of the die it takes on a good week, not because there's a multi-day
recovery process running down.

## 6. Sleep duration: a plateau, not a line

Binned on sober nights only, so the alcohol effect doesn't blur the sleep-dose relationship.
Returns flatten hard after 8.5h — this is the shape the sleep-hours slider is built to show.

In [1]:
from pipeline.bins import build_bins_report
bins = build_bins_report()

pd.DataFrame(bins['sleep_hours_sober'])

   bin   n  mean_hrv  mean_recovery  pct_recovery_ge_80
  <=5h   2      26.0            4.5                 0.0
  5-6h   4     108.2           53.8                 0.0
6-6.5h   6     129.2           73.0                33.3
6.5-7h  21     128.3           71.8                38.1
7-7.5h  47     124.6           72.3                38.3
7.5-8h 100     128.1           72.9                43.0
8-8.5h  94     130.9           76.5                48.9
8.5-9h  52     128.7           75.4                48.1
   9h+  60     127.0           72.7                46.7

### Respiratory rate, for context (signal, not a slider — see above)

In [1]:
pd.DataFrame(bins['resp_rate_sober'])

    bin   n  mean_hrv  mean_recovery  pct_recovery_ge_80
 <=15.0  24     148.7           78.8                50.0
15-15.5  63     127.7           73.8                46.0
15.5-16 144     128.2           75.1                46.5
  16-17 149     125.1           71.9                40.3
    >17   6      87.2           45.5                33.3

## 7. Workout type → next day's HRV (correlational panel, not a lever)

Presented as correlational in the app: she likely picks a gentle workout on a day she already
feels good, which would produce exactly this ranking without any causal effect of the workout
itself.

In [1]:
pd.DataFrame(bins['workout_next_day_hrv']['activities'])

Baseline (all days): 117.1 ms

        activity   n  mean_next_day_hrv
         Pilates  47              132.5
            Yoga  52              126.2
            HIIT  72              121.9
Strength Trainer  21              120.2
        Rest day 273              115.5
            Spin 191              113.9
         Walking  79              113.1
        Activity  39              110.1

## 8. High-HRV days vs low-HRV days

The single most quotable number in the project: how rarely a best-HRV day follows a drink,
compared to a worst-HRV day.

In [1]:
model['hrv_quartile_comparison']

{
  "high_hrv_threshold": 139.0,
  "low_hrv_threshold": 98.0,
  "high": {
    "n": 210,
    "sleep_hours": 8.2,
    "resp_rate": 15.7,
    "sleep_consistency_pct": 71.1,
    "alcohol_n": 7,
    "alcohol_known_n": 134
  },
  "low": {
    "n": 209,
    "sleep_hours": 7.7,
    "resp_rate": 16.1,
    "sleep_consistency_pct": 67.5,
    "alcohol_n": 78,
    "alcohol_known_n": 123
  }
}

## The honest ceiling

R² ≈ 0.26 for the fitted model. Roughly three-quarters of day-to-day HRV variance is
unexplained by anything in this export — stress, illness, travel, alcohol consumed off-journal,
and plenty else. The app states this ceiling directly in the UI rather than implying the model
knows more than it does.